#English to Hindi translation using Transformer architecture, with a Streamlit frontend

In [8]:
%%writefile requirements.txt

tensorflow
streamlit
numpy
scikit-learn


Writing requirements.txt


In [2]:
%%writefile transformer.py


import tensorflow as tf
from tensorflow.keras.layers import Embedding, Dense, LayerNormalization, Dropout
import numpy as np

class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, position, d_model):
        super().__init__()
        angle_rads = self.get_angles(np.arange(position)[:, np.newaxis],
                                     np.arange(d_model)[np.newaxis, :],
                                     d_model)
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        self.pos_encoding = tf.cast(angle_rads[np.newaxis, ...], dtype=tf.float32)

    def get_angles(self, pos, i, d_model):
        angle_rates = 1 / np.power(10000, (2 * (i//2)) / np.float32(d_model))
        return pos * angle_rates

    def call(self, x):
        return x + self.pos_encoding[:, :tf.shape(x)[1], :]

def transformer_encoder(num_layers, d_model, num_heads, dff, input_vocab_size, maximum_position_encoding):
    inputs = tf.keras.Input(shape=(None,))
    padding_mask = tf.keras.Input(shape=(1, 1, None))

    x = Embedding(input_vocab_size, d_model)(inputs)
    x = PositionalEncoding(maximum_position_encoding, d_model)(x)

    for _ in range(num_layers):
        attn_output = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(x, x, attention_mask=padding_mask)
        x = LayerNormalization(epsilon=1e-6)(x + attn_output)
        ffn_output = Dense(dff, activation='relu')(x)
        ffn_output = Dense(d_model)(ffn_output)
        x = LayerNormalization(epsilon=1e-6)(x + ffn_output)

    return tf.keras.Model(inputs=[inputs, padding_mask], outputs=x)


Writing transformer.py


In [3]:
%%writefile tokenizer.py

# model/tokenizer.py
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

class CustomTokenizer:
    def __init__(self, num_words=2000):
        self.tokenizer = Tokenizer(num_words=num_words, oov_token='<OOV>')

    def fit(self, texts):
        self.tokenizer.fit_on_texts(texts)

    def texts_to_seqs(self, texts):
        return self.tokenizer.texts_to_sequences(texts)

    def pad_sequences(self, seqs, maxlen=None):
        return pad_sequences(seqs, maxlen=maxlen, padding='post')

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump(self.tokenizer, f)

    def load(self, path):
        with open(path, 'rb') as f:
            self.tokenizer = pickle.load(f)


Writing tokenizer.py


In [4]:
%%writefile data_loader.py

# utils/data_loader.py
def load_data():
    # Small synthetic dataset
    en = ["Hello", "How are you", "What is your name", "Good morning", "Thank you"]
    hi = ["नमस्ते", "आप कैसे हैं", "आपका नाम क्या है", "सुप्रभात", "धन्यवाद"]
    return en, hi


Writing data_loader.py


In [5]:
%%writefile train.py

# model/train.py
from utils.data_loader import load_data
from model.tokenizer import CustomTokenizer
import tensorflow as tf
import os

def train_model():
    en_texts, hi_texts = load_data()

    en_tokenizer = CustomTokenizer()
    hi_tokenizer = CustomTokenizer()
    en_tokenizer.fit(en_texts)
    hi_tokenizer.fit(hi_texts)

    # Tokenize and pad
    en_seqs = en_tokenizer.texts_to_seqs(en_texts)
    hi_seqs = hi_tokenizer.texts_to_seqs(hi_texts)

    max_len = 5
    en_seqs = en_tokenizer.pad_sequences(en_seqs, maxlen=max_len)
    hi_seqs = hi_tokenizer.pad_sequences(hi_seqs, maxlen=max_len)

    vocab_size = 2000

    # Build a simple Seq2Seq model
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),
        tf.keras.layers.LSTM(128, return_sequences=True),
        tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(vocab_size, activation='softmax'))
    ])

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Reshape target: (samples, sequence_len, 1)
    hi_seqs = hi_seqs.reshape((hi_seqs.shape[0], hi_seqs.shape[1], 1))

    model.fit(en_seqs, hi_seqs, epochs=300, verbose=1)

    if not os.path.exists("checkpoints"):
        os.mkdir("checkpoints")
    model.save("checkpoints/model.h5")
    en_tokenizer.save("checkpoints/en_tokenizer.pkl")
    hi_tokenizer.save("checkpoints/hi_tokenizer.pkl")




Writing train.py


In [6]:
%%writefile translate.py

# utils/translate.py
import tensorflow as tf
from model.tokenizer import CustomTokenizer
import numpy as np

def translate(input_text):
    model = tf.keras.models.load_model("checkpoints/model.h5")
    en_tokenizer = CustomTokenizer(); en_tokenizer.load("checkpoints/en_tokenizer.pkl")
    hi_tokenizer = CustomTokenizer(); hi_tokenizer.load("checkpoints/hi_tokenizer.pkl")

    seq = en_tokenizer.texts_to_seqs([input_text])
    seq = en_tokenizer.pad_sequences(seq, maxlen=5)
    pred = model.predict(seq)

    pred_tokens = np.argmax(pred[0], axis=-1)
    inv_vocab = {v: k for k, v in hi_tokenizer.tokenizer.word_index.items()}

    return ' '.join([inv_vocab.get(tok, '') for tok in pred_tokens if tok != 0])


Writing translate.py


In [7]:
%%writefile app.py

# app.py
import streamlit as st
from utils.translate import translate
from model.train import train_model
import os

st.set_page_config(page_title="English to Hindi Translator", layout="centered")

st.title("🧠 English to Hindi Translator (Transformer-based)")
input_text = st.text_input("Enter English sentence:", "")

if st.button("Translate"):
    if not os.path.exists("checkpoints/model.h5"):
        with st.spinner("Training the model, please wait..."):
            train_model()
    result = translate(input_text)
    st.success(f"🔁 Hindi Translation: {result}")


Writing app.py
